2.2

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    对文本进行预处理，构建词汇表并生成自回归语言模型的滑动窗口样本。

    参数:
        text (str): 输入原始文本
        n (int): 滑动窗口长度（特征词数）

    返回:
        vocab (dict): 词汇表字典，键为词，值为从0开始的整数ID（按频率降序排列）
        (features, labels) (tuple): features为二维列表（每个元素是长度为n的词列表），
                                    labels为一维列表（每个元素为下一个词或None）
    """
    # 1. 转小写，并去除所有非字母和非空格的字符（保留空格用于分词）
    cleaned = re.sub(r'[^a-z\s]', '', text.lower())
    
    # 2. 按空白字符分词（自动处理多余空格）
    words = cleaned.split()
    
    # 如果文本没有有效词，返回空结果
    if not words:
        return {}, ([], [])
    
    # 3. 统计词频，并按频率降序排序（频率相同按字母升序），分配ID
    freq = Counter(words)
    sorted_words = sorted(freq.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    total_len = len(words)
    
    # 窗口起始位置范围：0 到 total_len - n（包含）
    for i in range(total_len - n + 1):
        feature = words[i:i+n]          # 长度为n的词列表
        # 如果窗口之后还有词，则作为标签，否则为None
        label = words[i+n] if i + n < total_len else None
        features.append(feature)
        labels.append(label)
    
    return vocab, (features, labels)

    # 执行示例
text = "The time machine"
n = 2
vocab, (feat, lab) = preprocess_text(text, n)

print("词汇表:", vocab)
print("特征:", feat)
print("标签:", lab)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time'], ['time', 'machine']]
标签: ['machine', None]


3.2

In [2]:
import numpy as np

def rnn_step_forward(x, h_prev, W_hx, W_hh, b_h):
    """
    RNN 单步前向传播。
    
    参数:
        x: 当前输入，形状 (batch_size, input_size)
        h_prev: 上一时刻隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏的权重，形状 (hidden_size, input_size)
        W_hh: 隐藏到隐藏的权重，形状 (hidden_size, hidden_size)
        b_h: 偏置，形状 (hidden_size,)
    
    返回:
        h_next: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 包含前向计算中间结果的元组，用于反向传播
    """
    # 线性组合：a = x * W_hx^T + h_prev * W_hh^T + b_h
    # 这里使用矩阵乘法，保证维度的正确性
    a = x @ W_hx.T + h_prev @ W_hh.T + b_h   # (batch, hidden)
    h_next = np.tanh(a)
    
    cache = (x, h_prev, W_hx, W_hh, b_h, a)
    return h_next, cache

def rnn_step_backward(dh_next, cache):
    """
    RNN 单步反向传播（仅计算梯度，不更新参数）。
    
    参数:
        dh_next: 损失对 h_next 的梯度，形状 (batch_size, hidden_size)
        cache: 前向传播时保存的元组 (x, h_prev, W_hx, W_hh, b_h, a)
    
    返回:
        dx: 损失对 x 的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对 h_prev 的梯度，形状 (batch_size, hidden_size)
        dW_hx: 损失对 W_hx 的梯度，形状 (hidden_size, input_size)
        dW_hh: 损失对 W_hh 的梯度，形状 (hidden_size, hidden_size)
        db_h: 损失对 b_h 的梯度，形状 (hidden_size,)
    """
    x, h_prev, W_hx, W_hh, b_h, a = cache
    batch_size = x.shape[0]
    
    # 1. 计算 da = dL/da, 其中 a = tanh^{-1}(h_next)
    da = dh_next * (1 - np.tanh(a) ** 2)   # (batch, hidden)
    
    # 2. 对偏置的梯度：对 batch 维求和
    db_h = np.sum(da, axis=0)              # (hidden,)
    
    # 3. 对权重的梯度：dW = da^T @ 输入
    dW_hx = da.T @ x                       # (hidden, input)
    dW_hh = da.T @ h_prev                  # (hidden, hidden)
    
    # 4. 对输入的梯度：dx = da @ W (注意 W 的形状)
    dx = da @ W_hx                         # (batch, input)
    
    # 5. 对上一隐藏状态的梯度：dh_prev = da @ W_hh
    dh_prev = da @ W_hh                    # (batch, hidden)
    
    return dx, dh_prev, dW_hx, dW_hh, db_h

# 设置随机种子
np.random.seed(0)

batch_size, input_size, hidden_size = 3, 4, 5
x = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(hidden_size, input_size)
W_hh = np.random.randn(hidden_size, hidden_size)
b_h = np.random.randn(hidden_size)

# 前向
h_next, cache = rnn_step_forward(x, h_prev, W_hx, W_hh, b_h)

# 模拟上游梯度
dh_next = np.random.randn(batch_size, hidden_size)

# 反向
dx, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)

print("dx shape:", dx.shape)          # (3, 4)
print("dh_prev shape:", dh_prev.shape) # (3, 5)
print("dW_hx shape:", dW_hx.shape)    # (5, 4)
print("dW_hh shape:", dW_hh.shape)    # (5, 5)
print("db_h shape:", db_h.shape)      # (5,)

dx shape: (3, 4)
dh_prev shape: (3, 5)
dW_hx shape: (5, 4)
dW_hh shape: (5, 5)
db_h shape: (5,)


4.2

In [3]:
import torch
import torch.nn as nn

def bidirectional_rnn_encoder(X, hidden_dim, num_layers=1):
    """
    双向 RNN 编码器

    参数:
        X: 输入序列，形状 (seq_len, batch, input_dim)
        hidden_dim: 每方向隐藏单元数
        num_layers: RNN 层数（默认 1）

    返回:
        outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
        final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
    """
    input_dim = X.size(2)

    # 创建双向 RNN 层（默认使用 tanh 激活）
    rnn = nn.RNN(
        input_size=input_dim,
        hidden_size=hidden_dim,
        num_layers=num_layers,
        bidirectional=True,
        batch_first=False          # 输入形状为 (seq, batch, feature)
    )

    outputs, _ = rnn(X)            # outputs: (seq_len, batch, num_directions * hidden_dim)

    # 取最后一个时间步的拼接输出
    final_state = outputs[-1]      # (batch, 2 * hidden_dim)

    return outputs, final_state

# 构造测试数据：序列长度 5，批次 3，输入维度 10
seq_len, batch, input_dim = 5, 3, 10
hidden_dim = 4
X = torch.randn(seq_len, batch, input_dim)

outputs, final_state = bidirectional_rnn_encoder(X, hidden_dim)

print("outputs shape:", outputs.shape)        # torch.Size([5, 3, 8])
print("final_state shape:", final_state.shape) # torch.Size([3, 8])

outputs shape: torch.Size([5, 3, 8])
final_state shape: torch.Size([3, 8])


5.2

In [4]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, target_indices, W, W_out):
    """
    CBOW 模型前向传播（完整 Softmax）

    参数:
        context_indices: 上下文词索引，形状 (batch_size, context_size)
        target_indices: 中心词索引，形状 (batch_size,)
        W: 输入嵌入矩阵，形状 (vocab_size, embedding_dim)
        W_out: 输出嵌入矩阵，形状 (embedding_dim, vocab_size)

    返回:
        loss: 交叉熵损失标量 (tensor)
    """
    # 1. 获取上下文词的嵌入向量
    # context_indices: (batch, context_size) -> 取 W 对应的行
    context_embeds = W[context_indices]          # (batch, context_size, embed_dim)

    # 2. 计算平均上下文向量（隐藏层）
    hidden = context_embeds.mean(dim=1)          # (batch, embed_dim)

    # 3. 计算输出 logits
    logits = hidden @ W_out                      # (batch, vocab_size)

    # 4. 计算交叉熵损失（内置 softmax）
    loss = F.cross_entropy(logits, target_indices)

    return loss


# 设置参数
batch_size = 4
context_size = 3
vocab_size = 10
embed_dim = 5

# 随机生成数据
context_indices = torch.randint(0, vocab_size, (batch_size, context_size))
target_indices = torch.randint(0, vocab_size, (batch_size,))
W = torch.randn(vocab_size, embed_dim, requires_grad=True)
W_out = torch.randn(embed_dim, vocab_size, requires_grad=True)

# 前向计算损失
loss = cbow_forward(context_indices, target_indices, W, W_out)
print("损失值:", loss.item())

# 反向传播可自动求导
loss.backward()
print("W 梯度形状:", W.grad.shape)      # (10, 5)
print("W_out 梯度形状:", W_out.grad.shape) # (5, 10)

损失值: 3.587120771408081
W 梯度形状: torch.Size([10, 5])
W_out 梯度形状: torch.Size([5, 10])


6.2

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """
    多头注意力模块（无掩码）
    - d_model: 模型维度（必须能被 num_heads 整除）
    - num_heads: 注意力头数
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads   # 每个头的键/查询维度
        self.d_v = d_model // num_heads   # 每个头的值维度

        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X):
        """
        输入:
            X: 形状 (seq_len, batch, d_model)
        输出:
            out: 形状 (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.size()

        # 1. 线性投影得到 Q, K, V，形状 (seq_len, batch, d_model)
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        # 2. 分割多头：将最后一维拆分为 (num_heads, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k)  # (seq, batch, heads, d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k)
        V = V.view(seq_len, batch, self.num_heads, self.d_v)

        # 3. 交换维度，便于批量计算： (batch, heads, seq, d_k)
        Q = Q.permute(1, 2, 0, 3)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)

        # 4. 缩放点积注意力
        # scores: (batch, heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)  # (batch, heads, seq, d_v)

        # 5. 恢复原始维度顺序并拼接多头
        attn_output = attn_output.permute(2, 0, 1, 3)  # (seq, batch, heads, d_v)
        concat = attn_output.contiguous().view(seq_len, batch, self.d_model)  # (seq, batch, d_model)

        # 6. 最终线性层
        output = self.W_o(concat)  # (seq, batch, d_model)
        return output
    

    # 参数设置
d_model = 4
num_heads = 2
seq_len = 3
batch = 2

# 随机输入
X = torch.randn(seq_len, batch, d_model)

# 创建多头注意力层
mha = MultiHeadAttention(d_model, num_heads)
output = mha(X)

print("输入形状:", X.shape)      # torch.Size([3, 2, 4])
print("输出形状:", output.shape)  # torch.Size([3, 2, 4])

输入形状: torch.Size([3, 2, 4])
输出形状: torch.Size([3, 2, 4])
